# M3 학습예산 진단 — candidate-specific N/V 엣지 가중

완료된 L2 `1e-3`·seed 42·300 epoch M1 곡선을 신분검사 후 재사용하고, **M3 한 arm만** 300 epoch까지 학습합니다. 25 epoch마다 같은 신규상품 개발분할을 평가합니다.

M3 엣지식은 기존 사양을 바꾸지 않습니다.

- 후보 적합도: `F(u,i)=1-(|q_N-b_N|+|q_V-p_V|)/2`
- 엣지계수: `exp[0.15·q_C·(F-user_mean_F)]`
- 사용자별 평균 계수 1로 정규화
- K=1 균일 음성, 손실가중 없음, 외부 재정렬 없음

단일 개발 seed의 학습곡선 진단이며 epoch 선택·유의성·안정성·CLV 귀속을 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '401f71d9e10ff7cd2f1afc208e3f81aee19450f9'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import json
import torch
import lightgcn_clv_m3_training_budget as m3

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
REFERENCE = '/content/drive/MyDrive/논문/data/results_v3_dunnhumby_clv_m2_capacity_search_v1/clv_m2_capacity_search_5f4b9c6e45d2_curve.csv'
OUT_DIR = '/content/drive/MyDrive/논문/data/results_v3_dunnhumby_clv_m3_candidate_nv_training_budget_v1'
cfg = m3.configure_m3_budget(reference_curve_csv=REFERENCE, out_dir=OUT_DIR)
summary = m3.preflight_summary(cfg)
assert summary['m3']['beta'] == 0.15
assert summary['fixed']['negative_sampling'] == 'one uniform unseen item'
assert summary['fixed']['final_test'] is False and summary['fixed']['holdout'] is False
reference = m3.load_reference_curve(cfg)  # 신분 불일치 시 여기서 중단
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('재사용 M1 평가 epoch:', reference.epoch.tolist())

In [ ]:
curve = m3.run_m3_budget(cfg)

In [ ]:
from IPython.display import display
print('1) M3 전체 학습곡선')
display(curve)
print('2) 동일 epoch M3 - M1')
display(curve.attrs['gap'])
print('저장 파일:', json.dumps(curve.attrs['result_paths'], ensure_ascii=False, indent=2))

In [ ]:
import matplotlib.pyplot as plt
ref = curve.attrs['reference']
gap = curve.attrs['gap']
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(ref.epoch, ref['recall@10'], marker='o', label='M1 reused')
axes[0].plot(curve.epoch, curve['recall@10'], marker='o', label='M3')
axes[1].plot(gap.epoch, gap['recall@10'], marker='o', label='M3-M1')
for ax in axes:
    ax.axvline(100, color='gray', linestyle='--'); ax.set_xlabel('epoch'); ax.legend()
axes[0].set_title('Development Recall@10'); axes[1].set_title('M3 - M1 Recall@10')
axes[1].axhline(0, color='black', linewidth=.8)
plt.tight_layout(); plt.show()